# DX 704 Week 9 Project

This week's project will build an email spam classifier based on the Enron email data set.
You will perform your own feature extraction, and use naive Bayes to estimate the probability that a particular email is spam or not.
Finally, you will review the tradeoffs from different thresholds for automatically sending emails to the junk folder.

The full project description and a template notebook are available on GitHub: [Project 9 Materials](https://github.com/bu-cds-dx704/dx704-project-09).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Download Data Set

We will be using the Enron spam data set as prepared in this GitHub repository.

https://github.com/MWiechmann/enron_spam_data

You may need to download this differently depending on your environment.

In [1]:
!wget https://github.com/MWiechmann/enron_spam_data/raw/refs/heads/master/enron_spam_data.zip

--2026-03-30 02:02:18--  https://github.com/MWiechmann/enron_spam_data/raw/refs/heads/master/enron_spam_data.zip
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/MWiechmann/enron_spam_data/refs/heads/master/enron_spam_data.zip [following]
--2026-03-30 02:02:18--  https://raw.githubusercontent.com/MWiechmann/enron_spam_data/refs/heads/master/enron_spam_data.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 15642124 (15M) [application/zip]
Saving to: ‘enron_spam_data.zip.2’

enron_spam_data.zip 100%[===================>]  14.92M  41.7MB/s    in 0.4s    

2026-03-30 02:02:18 (41.7 MB/s) - ‘enron_s

In [2]:
!pip install pandas

import pandas as pd
import numpy as np
import re
import json


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [3]:
# pandas can read the zip file directly
enron_spam_data = pd.read_csv("enron_spam_data.zip")
enron_spam_data
import re

def extract_features(subject, message):
    text = str(subject) + " " + str(message)
    text = text.lower()
    
    words = re.findall(r"[a-z0-9']+", text)
    
    features = {}
    for word in words:
        if len(word) >= 3:
            features[word] = features.get(word, 0) + 1
            
    return features

In [4]:
(enron_spam_data["Spam/Ham"] == "spam").mean()

np.float64(0.5092834262664611)

## Part 2: Design a Feature Extractor

Design a feature extractor for this data set and write out two files of features based on the text.
Don't forget that both the Subject and Message columns are relevant sources of text data.
For each email, you should count the number of repetitions of each feature present.
The auto-grader will assume that you are using a multinomial distribution in the following problems.

In [5]:
# YOUR CHANGES HERE

enron_spam_data["features"] = enron_spam_data.apply(
    lambda row: extract_features(row["Subject"], row["Message"]),
    axis=1
)

train_data = enron_spam_data[enron_spam_data["Message ID"] % 30 != 0].copy()
test_data = enron_spam_data[enron_spam_data["Message ID"] % 30 == 0].copy()

train_output = train_data[["Message ID", "features"]].copy()
train_output["features_json"] = train_output["features"].apply(json.dumps)
train_output = train_output[["Message ID", "features_json"]]
train_output.to_csv("train-features.tsv", sep="\t", index=False)

test_output = test_data[["Message ID", "features"]].copy()
test_output["features_json"] = test_output["features"].apply(json.dumps)
test_output = test_output[["Message ID", "features_json"]]
test_output.to_csv("test-features.tsv", sep="\t", index=False)

Assign a row to the test data set if `Message ID % 30 == 0` and assign it to the training data set otherwise.
Write two files, "train-features.tsv" and "test-features.tsv" with two columns, Message ID and features_json.
The features_json column should contain a JSON dictionary where the keys are your feature names and the values are integer feature values.
This will give us a sparse feature representation.


In [6]:
# YOUR CHANGES HERE

enron_spam_data["features"] = enron_spam_data.apply(
    lambda row: extract_features(row["Subject"], row["Message"]),
    axis=1
)

train_data = enron_spam_data[enron_spam_data["Message ID"] % 30 != 0].copy()
test_data = enron_spam_data[enron_spam_data["Message ID"] % 30 == 0].copy()

train_output = train_data[["Message ID", "features"]].copy()
train_output["features_json"] = train_output["features"].apply(json.dumps)
train_output = train_output[["Message ID", "features_json"]]
train_output.to_csv("train-features.tsv", sep="\t", index=False)

test_output = test_data[["Message ID", "features"]].copy()
test_output["features_json"] = test_output["features"].apply(json.dumps)
test_output = test_output[["Message ID", "features_json"]]
test_output.to_csv("test-features.tsv", sep="\t", index=False)

Submit "train-features.tsv" and "test-features.tsv" in Gradescope.

Hint: these features will be graded based on the test accuracy of a logistic regression based on the training features.
This is to make sure that your feature set is not degenerate; you do not need to compute this regression yourself.
You can separately assess your feature quality based on your results in part 6.

## Part 3: Compute Conditional Probabilities

Based on your training data, compute appropriate conditional probabilities for use with naïve Bayes.
Use of additive smoothing with $\alpha=1$ to avoid zeros.


In [7]:
# YOUR CHANGES HERE

from collections import defaultdict

spam_counts = defaultdict(int)
ham_counts = defaultdict(int)

spam_total = 0
ham_total = 0

for _, row in train_data.iterrows():
    label = row["Spam/Ham"]
    features = row["features"]
    
    for word, count in features.items():
        if label == "spam":
            spam_counts[word] += count
            spam_total += count
        else:
            ham_counts[word] += count
            ham_total += count

Save the conditional probabilities in a file "feature-probabilities.tsv" with columns feature, ham_probability and spam_probability.

In [8]:
# YOUR CHANGES HERE

# build vocabulary
vocab = set(list(spam_counts.keys()) + list(ham_counts.keys()))
V = len(vocab)

alpha = 1

feature_probs = []

for word in vocab:
    spam_prob = (spam_counts[word] + alpha) / (spam_total + alpha * V)
    ham_prob  = (ham_counts[word] + alpha) / (ham_total + alpha * V)
    
    feature_probs.append((word, ham_prob, spam_prob))

In [9]:
with open("feature-probabilities.tsv", "w") as f:
    f.write("feature\tham_probability\tspam_probability\n")
    
    for word, ham_p, spam_p in feature_probs:
        f.write(f"{word}\t{ham_p}\t{spam_p}\n")

Submit "feature-probabilities.tsv" in Gradescope.

## Part 4: Implement a Naïve Bayes Classifier

Implement a naïve Bayes classifier based on your previous feature probabilities.

In [10]:
# YOUR CHANGES HERE

import math

spam_prior = (train_data["Spam/Ham"] == "spam").mean()
ham_prior = 1 - spam_prior

prob_dict = {word: (ham_p, spam_p) for word, ham_p, spam_p in feature_probs}

def predict_naive_bayes(features):
    log_ham = math.log(ham_prior)
    log_spam = math.log(spam_prior)
    
    for word, count in features.items():
        if word in prob_dict:
            ham_p, spam_p = prob_dict[word]
            log_ham += count * math.log(ham_p)
            log_spam += count * math.log(spam_p)
    
    max_log = max(log_ham, log_spam)
    ham_score = math.exp(log_ham - max_log)
    spam_score = math.exp(log_spam - max_log)
    
    total = ham_score + spam_score
    ham_probability = ham_score / total
    spam_probability = spam_score / total
    
    return ham_probability, spam_probability

Save your prediction probabilities to "train-predictions.tsv" with columns Message ID, ham and spam.

In [11]:
# YOUR CHANGES HERE

with open("train-predictions.tsv", "w") as f:
    f.write("Message ID\tham\tspam\n")
    
    for _, row in train_data.iterrows():
        ham_p, spam_p = predict_naive_bayes(row["features"])
        f.write(f"{row['Message ID']}\t{ham_p}\t{spam_p}\n")

Submit "train-predictions.tsv" in Gradescope.

## Part 5: Predict Spam Probability for Test Data

Use your previous classifier to predict spam probability for the test data.
If the test data has tokens that you did not model, just ignore them.

In [12]:
# YOUR CHANGES HERE

with open("test-predictions.tsv", "w") as f:
    f.write("Message ID\tham\tspam\n")
    
    for _, row in test_data.iterrows():
        ham_p, spam_p = predict_naive_bayes(row["features"])
        f.write(f"{row['Message ID']}\t{ham_p}\t{spam_p}\n")

Save your prediction probabilities in "test-predictions.tsv" with the same columns as "train-predictions.tsv".

In [13]:
# YOUR CHANGES HERE

with open("test-predictions.tsv", "w") as f:
    f.write("Message ID\tham\tspam\n")
    
    for _, row in test_data.iterrows():
        ham_p, spam_p = predict_naive_bayes(row["features"])
        f.write(f"{row['Message ID']}\t{ham_p}\t{spam_p}\n")

Submit "test-predictions.tsv" in Gradescope.

## Part 6: Construct ROC Curve

For every probability threshold from 0.01 to .99 in increments of 0.01, compute the false and true positive rates from the test data using the spam class for positives.
That is, if the predicted spam probability is greater than or equal to the threshold, predict spam.

In [14]:
# YOUR CHANGES HERE

thresholds = np.arange(0.01, 1.00, 0.01)

roc_results = []

for threshold in thresholds:
    tp = 0
    fp = 0
    tn = 0
    fn = 0
    
    for _, row in test_data.iterrows():
        ham_p, spam_p = predict_naive_bayes(row["features"])
        predicted_label = "spam" if spam_p >= threshold else "ham"
        actual_label = row["Spam/Ham"]
        
        if predicted_label == "spam" and actual_label == "spam":
            tp += 1
        elif predicted_label == "spam" and actual_label == "ham":
            fp += 1
        elif predicted_label == "ham" and actual_label == "ham":
            tn += 1
        elif predicted_label == "ham" and actual_label == "spam":
            fn += 1
    
    false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0
    true_positive_rate = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    roc_results.append((threshold, false_positive_rate, true_positive_rate))

Save this data in a file "roc.tsv" with columns threshold, false_positive_rate and true_positive rate.

In [15]:
# YOUR CHANGES HERE

with open("roc.tsv", "w") as f:
    f.write("threshold\tfalse_positive_rate\ttrue_positive_rate\n")
    
    for t, fpr, tpr in roc_results:
        f.write(f"{t}\t{fpr}\t{tpr}\n")

Submit "roc.tsv" in Gradescope.

## Part 7: Signup for Gemini API Key

Create a free Gemini API key at https://aistudio.google.com/app/api-keys.
You will need to do this with a personal Google account - it will not work with your BU Google account.
This will not incur any charges unless you configure billing information for the key.

You will be asked to start a Gemini free trial for week 11.
This will not incur any charges unless you exceed expected usage by an order of magnitude.


No submission needed.

## Part 8: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 9: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.

In [16]:
with open("acknowledgments.txt", "w") as f:
    f.write("""I completed this assignment independently and did not discuss it with others.

I did not use any libraries beyond those provided in the course materials.

I used ChatGPT as a learning aid to help debug code, understand Naïve Bayes concepts, and structure parts of the implementation. All submitted work reflects my understanding of the material.""")